# Testing Claim 1 on an open-data CPU surrogate

## 0. Motivation

The paper studies frozen medical foundation-model embeddings and reports a quantum-support-vector-machine (QSVM) advantage for detecting the minority class. The primary metric is minority-class F1: it is high only when the model finds minority examples with both useful precision and useful recall.

This notebook asks a narrower question: **does the direction and consistency of the paper's Claim 1 appear in our CPU-only PneumoniaMNIST surrogate?** It compares the preserved upstream QSVM with a linear SVM and a validation-tuned RBF SVM on identical local samples and splits.

MerLin is included as a fourth, data-paired model. It uses the same N=500 subset, split seed, and PCA dimension as the other models. However, its photonic feature map is not the paper's qubit BSP map, so it is discussed separately as an adaptation rather than as evidence for Claim 1.

This notebook loads small curated artifacts only. It does not recompute kernels, train models, or include medical images.

## 1. Claim 1 in the paper

On its MIMIC-CXR insurance task and frozen medical embeddings, the [paper](https://arxiv.org/abs/2604.24597v1) reports that:

- a fidelity/BSP QSVM with `C=1`, one circuit repetition, and trace normalization has higher minority-class F1 than an equally untuned linear SVM with `C=1` for all 18 model–qubit pairs actually tested, with qubit counts drawn from `{4, 6, 8, 9, 10, 11, 12, 16}`;
- across ten embedding seeds, 17 paired comparisons have `p<0.001` and one has `p<0.01` in the paper's paired bootstrap analysis;
- the linear SVM has minority-class F1 equal to zero on 90–100% of seeds at every tested qubit count;
- against the best validation-tuned RBF SVM at the same PCA dimension, the QSVM wins all seven reported configurations, with a reported mean gain of `+0.068`.

A configuration-level win compares aggregate results across seeds; it does not necessarily mean that QSVM wins every individual seed. Our local checks are therefore:

1. Is the mean paired difference `F1(QSVM) - F1(baseline)` positive at q=4 and q=6?
2. How often does QSVM win, tie, or lose for the same local seed?
3. Does the local linear SVM collapse to minority F1 equal to zero?

The paper's seeds generate different embeddings. Our seeds instead control subsampling and the train/validation/test split, so the two protocols are not statistically equivalent.

MerLin does not enter these Claim 1 checks. Section 4 asks a separate descriptive question: how does a matched photonic fidelity kernel behave under the same local data protocol?

## 2. Our setup

| Item | Local adaptation |
|---|---|
| Scope | Open-data CPU surrogate, not the reference reproduction |
| Dataset | Official PneumoniaMNIST training split |
| Representation | Raw flattened 28×28 pixels, not frozen embeddings |
| Sample count | N=500 for each seed |
| Split | 400 train / 50 validation / 50 test, stratified |
| Minority class | `normal`, with 11–14 test examples depending on the seed |
| Seeds | 0–9, controlling both subsampling and splitting |
| Dimensions | PCA dimension q=4 and q=6; the QSVM uses the same number of qubits |
| QSVM | Preserved upstream BSP path, `C=1`, one repetition, CPU, historical trace path |
| Linear baseline | Linear SVM, `C=1` |
| RBF baseline | `C` selected from `{0.01, 0.1, 1, 10, 100}` using validation minority F1 |
| MerLin adaptation | Photonic fidelity kernel, `C=1`, exact CPU simulation, fixed circuit seed 0 |
| Primary metric | Test F1 for the `normal` class |

A comparison is data-paired when q and `data_seed` are identical. All four models then use the same N=500 subset and train/validation/test split. MerLin's circuit seed remains fixed at 0, so the ten repetitions measure data-and-split variation only. The curated artifact contains 4 models × 2 dimensions × 10 seeds = 80 records.

The raw run was written under `outdir/n500-q4-q6/` by `scripts/run_table1_adaptation.sh`. `results/q4_q6_n500_per_seed.csv` was curated from those outputs and adds the dataset checksum and split identifiers. It contains metrics and metadata only; it is not generated automatically by the current pipeline.

> **Diagnostic protocol:** the initial upstream behavior is deliberately preserved. The final MinMax preprocessing in all four paths uses held-out data. In addition, the QSVM historical trace path divides square training Gram matrices by their trace while leaving rectangular validation/test cross-kernels unscaled. The audited circuit also differs from the paper's written BSP description. These deviations prevent a clean quantum-advantage claim.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

results_dir = Path.cwd() / 'results'
benchmark_path = results_dir / 'q4_q6_n500_per_seed.csv'
if not benchmark_path.is_file():
    raise FileNotFoundError('Run this notebook from the qsvm_medimage directory.')

benchmark = pd.read_csv(benchmark_path)
claim_model_order = ['qsvm', 'linear_svm', 'rbf_svm']
model_order = claim_model_order + ['merlin_fidelity']
model_labels = {
    'qsvm': 'QSVM',
    'linear_svm': 'Linear SVM',
    'rbf_svm': 'Tuned RBF SVM',
    'merlin_fidelity': 'MerLin fidelity',
}
keys = ['pca_dim', 'data_seed', 'model']
if benchmark.duplicated(keys).any():
    raise ValueError('Duplicate benchmark configurations found.')

expected = {
    (q, seed, model)
    for q in (4, 6)
    for seed in range(10)
    for model in model_order
}
observed = set(benchmark[keys].itertuples(index=False, name=None))
if observed != expected:
    raise ValueError('The N=500 benchmark is incomplete or contains unexpected rows.')
if not benchmark.groupby(['pca_dim', 'data_seed'])['split_id'].nunique().eq(1).all():
    raise ValueError('Compared configurations do not use identical splits.')
if benchmark['dataset_sha256'].nunique() != 1:
    raise ValueError('More than one prepared dataset is present.')
merlin_rows = benchmark.query("model == 'merlin_fidelity'")
if not (merlin_rows['protocol'].eq('fixed_circuit_seed_0').all() and merlin_rows['svc_c'].eq(1).all()):
    raise ValueError('Unexpected MerLin circuit seed or SVM C value.')

metric_columns = [
    'train_minority_f1',
    'val_minority_f1',
    'test_minority_f1',
    'test_accuracy',
    'test_auc',
]
if not np.isfinite(benchmark[metric_columns].to_numpy()).all():
    raise ValueError('The curated artifact contains non-finite metrics.')

claim_results = benchmark[benchmark['model'].isin(claim_model_order)].copy()
merlin_results = merlin_rows.copy()
print('Validated: 80 complete, unique, data-paired records.')
preview_columns = [
    'data_seed',
    'model',
    'test_minority_samples',
    'svc_c',
    'c_selection',
    'kernel_normalization',
    'test_minority_f1',
    'test_accuracy',
    'test_auc',
]
preview = benchmark.query('pca_dim == 4 and data_seed == 0')[preview_columns].copy()
preview['model'] = preview['model'].map(model_labels)
preview.rename(
    columns={
        'data_seed': 'seed',
        'test_minority_samples': 'minority test n',
        'svc_c': 'C',
        'c_selection': 'C selection',
        'kernel_normalization': 'kernel scaling',
        'test_minority_f1': 'minority F1',
        'test_accuracy': 'accuracy',
        'test_auc': 'AUC',
    }
).round(3)

## 3. Results and discussion

The Claim 1 analysis below uses the 60 QSVM, linear-SVM, and RBF-SVM records from the complete 80-record artifact. The matched MerLin results are reserved for Section 4. The uncertainty is the sample standard deviation across ten local seeds.

W/T/L means wins/ties/losses for QSVM against a baseline, evaluated separately on each identical data seed. It measures seed-level stability; it is not the paper's count of model–qubit configurations.

In [ ]:
summary = (
    claim_results.assign(
        train_zero=claim_results['train_minority_f1'].eq(0),
        test_zero=claim_results['test_minority_f1'].eq(0),
    )
    .groupby(['pca_dim', 'model'], as_index=False)
    .agg(
        n_seeds=('data_seed', 'nunique'),
        f1_mean=('test_minority_f1', 'mean'),
        f1_std=('test_minority_f1', 'std'),
        accuracy_mean=('test_accuracy', 'mean'),
        auc_mean=('test_auc', 'mean'),
        train_zero_seeds=('train_zero', 'sum'),
        test_zero_seeds=('test_zero', 'sum'),
    )
)
summary['model_order'] = pd.Categorical(
    summary['model'], categories=claim_model_order, ordered=True
)
summary = summary.sort_values(['pca_dim', 'model_order'])
summary['model_label'] = summary['model'].map(model_labels)

summary_table = summary[
    [
        'pca_dim',
        'model_label',
        'f1_mean',
        'f1_std',
        'test_zero_seeds',
        'train_zero_seeds',
        'accuracy_mean',
        'auc_mean',
    ]
].rename(
    columns={
        'pca_dim': 'q',
        'model_label': 'model',
        'f1_mean': 'minority F1 mean',
        'f1_std': 'minority F1 std',
        'test_zero_seeds': 'test F1=0 seeds',
        'train_zero_seeds': 'train F1=0 seeds',
        'accuracy_mean': 'accuracy mean',
        'auc_mean': 'AUC mean',
    }
)
summary_table.round(3)

In [ ]:
paired = claim_results.pivot(
    index=['pca_dim', 'data_seed'],
    columns='model',
    values='test_minority_f1',
).sort_index()

comparison_specs = [
    ('linear_svm', 'QSVM - linear SVM'),
    ('rbf_svm', 'QSVM - tuned RBF'),
]
paired_rows = []
delta_rows = []
tolerance = 1e-12
for q in (4, 6):
    q_values = paired.loc[q]
    for baseline, label in comparison_specs:
        delta = q_values['qsvm'] - q_values[baseline]
        paired_rows.append(
            {
                'q': q,
                'comparison': label,
                'mean delta': delta.mean(),
                'wins': int((delta > tolerance).sum()),
                'ties': int((delta.abs() <= tolerance).sum()),
                'losses': int((delta < -tolerance).sum()),
            }
        )
        for seed, value in delta.items():
            delta_rows.append(
                {'q': q, 'seed': seed, 'comparison': label, 'delta': value}
            )

paired_summary = pd.DataFrame(paired_rows)
paired_deltas = pd.DataFrame(delta_rows)
display(paired_summary.round({'mean delta': 3}))

rbf_c_counts = (
    claim_results.query("model == 'rbf_svm'")
    .groupby(['pca_dim', 'svc_c'])
    .size()
    .unstack(fill_value=0)
)
rbf_c_counts.index.name = 'q'
rbf_c_counts.columns.name = 'selected C'
display(rbf_c_counts)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))

q_values = np.array([4, 6])
x_positions = np.arange(len(q_values))
bar_width = 0.24
for model_index, model in enumerate(claim_model_order):
    model_summary = summary.query('model == @model').set_index('pca_dim').loc[q_values]
    axes[0].bar(
        x_positions + (model_index - 1) * bar_width,
        model_summary['f1_mean'],
        width=bar_width,
        yerr=model_summary['f1_std'],
        capsize=4,
        label=model_labels[model],
    )
axes[0].set_xticks(x_positions, [f'q={q}' for q in q_values])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Test minority-class F1')
axes[0].set_title('Mean ± standard deviation across seeds')
axes[0].legend()

for (q, comparison), group in paired_deltas.groupby(['q', 'comparison']):
    axes[1].plot(
        group['seed'],
        group['delta'],
        marker='o',
        label=f'q={q}: {comparison}',
    )
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(range(10))
axes[1].set_xlabel('Data and split seed')
axes[1].set_ylabel('Paired minority-F1 difference')
axes[1].set_title('Positive values favor QSVM')
axes[1].legend(fontsize=8)

figure.suptitle('PneumoniaMNIST raw-pixel surrogate, N=500')
figure.tight_layout()

### What the local result says

- At the configuration-mean level, QSVM is higher for both q values against the linear SVM and both q values against the tuned RBF. This is a descriptive `2/2` in each local comparison, not a reproduction of the paper's `18/18` or `7/7` results.
- Against the linear SVM, QSVM records 7 wins and 3 losses at both q=4 and q=6. Against RBF, it records 6/1/3 at q=4 but only 4/1/5 at q=6. The q=6 mean advantage over RBF is only `+0.003`.
- No model has test minority F1 equal to zero on any seed. The paper's linear-collapse pattern is therefore absent from this surrogate.
- The QSVM does not lead on every metric. Its mean test AUC is lower than both classical baselines at q=4 and q=6, and its q=6 mean accuracy is also lower.

A major diagnostic anomaly is visible in the summary: QSVM has training minority F1 equal to zero on all 20 q–seed runs, while its validation and test F1 values are nonzero. The code confirms that the square training kernel is trace-scaled while rectangular cross-kernels are not. This scale mismatch is consistent with the unusual train/test behavior, although this preserved benchmark does not isolate its causal effect.

Each N=500 test split contains 50 images. Nevertheless, no local confidence interval or significance test was predeclared, the local seeds are not embedding seeds, and the upstream protocol contains known deviations. The safe conclusion is descriptive: **the average F1 direction agrees with Claim 1 in these two surrogate configurations, but the reported systematic advantage and classical collapse are not reproduced.**

## 4. Matched MerLin comparison

For every q–seed pair, MerLin receives the same N=500 subset, 400/50/50 split, and PCA dimension as the three Claim 1 models. The data seed varies from 0 to 9, while the MerLin circuit seed remains fixed at 0.

| Property | Matched setting |
|---|---|
| Samples | N=500 |
| Dimensions | PCA q=4 and q=6 |
| Data/split seeds | 0–9 |
| Classifier | Precomputed-kernel SVM, `C=1` |
| Photonic resources at q=4 | 5 modes, 3 photons, input `[1, 0, 1, 0, 1]` |
| Photonic resources at q=6 | 7 modes, 4 photons, input `[1, 0, 1, 0, 1, 0, 1]` |
| Circuit seed | 0, fixed |

This removes sample-and-split differences from the local comparison. It does not make MerLin equivalent to the paper's BSP QSVM: the feature maps and physical models remain different.

In [ ]:
all_summary = (
    benchmark.groupby(['pca_dim', 'model'], as_index=False)
    .agg(
        f1_mean=('test_minority_f1', 'mean'),
        f1_std=('test_minority_f1', 'std'),
        accuracy_mean=('test_accuracy', 'mean'),
        auc_mean=('test_auc', 'mean'),
    )
)
all_summary['model_order'] = pd.Categorical(
    all_summary['model'], categories=model_order, ordered=True
)
all_summary = all_summary.sort_values(['pca_dim', 'model_order'])
all_summary['model'] = all_summary['model'].map(model_labels)
display(
    all_summary.drop(columns='model_order')
    .rename(
        columns={
            'pca_dim': 'q',
            'f1_mean': 'minority F1 mean',
            'f1_std': 'minority F1 std',
            'accuracy_mean': 'accuracy mean',
            'auc_mean': 'AUC mean',
        }
    )
    .round(3)
)

all_paired = benchmark.pivot(
    index=['pca_dim', 'data_seed'],
    columns='model',
    values='test_minority_f1',
).sort_index()
merlin_comparisons = [
    ('qsvm', 'MerLin - QSVM'),
    ('linear_svm', 'MerLin - linear SVM'),
    ('rbf_svm', 'MerLin - tuned RBF'),
]
merlin_paired_rows = []
for q in (4, 6):
    q_slice = all_paired.loc[q]
    for baseline, label in merlin_comparisons:
        delta = q_slice['merlin_fidelity'] - q_slice[baseline]
        merlin_paired_rows.append(
            {
                'q': q,
                'comparison': label,
                'mean delta': delta.mean(),
                'wins': int((delta > tolerance).sum()),
                'ties': int((delta.abs() <= tolerance).sum()),
                'losses': int((delta < -tolerance).sum()),
            }
        )
display(pd.DataFrame(merlin_paired_rows).round({'mean delta': 3}))

figure, axis = plt.subplots(figsize=(9, 4.5))
bar_width = 0.18
for model_index, model in enumerate(model_order):
    model_summary = (
        benchmark.query('model == @model')
        .groupby('pca_dim')['test_minority_f1']
        .agg(['mean', 'std'])
        .loc[q_values]
    )
    axis.bar(
        x_positions + (model_index - 1.5) * bar_width,
        model_summary['mean'],
        width=bar_width,
        yerr=model_summary['std'],
        capsize=4,
        label=model_labels[model],
    )
axis.set_xticks(x_positions, [f'q={q}' for q in q_values])
axis.set_ylim(0, 1.05)
axis.set_ylabel('Test minority-class F1')
axis.set_title('Data-paired N=500 comparison: mean ± standard deviation')
axis.legend()
figure.tight_layout()

The four-model table is a fair local comparison with respect to data: every score at a given q and seed comes from the same subset and split. The paired-difference table is written from MerLin's perspective, so a positive value favors MerLin.

- MerLin's mean minority F1 is `0.758 ± 0.132` at q=4 and `0.770 ± 0.096` at q=6.
- Against QSVM, its mean differences are `−0.051` at q=4 (4/0/6 W/T/L) and `−0.052` at q=6 (3/0/7).
- Against the linear SVM, it is nearly tied on average at q=4 (`−0.0002`) and lower at q=6 (`−0.016`).
- Against tuned RBF, it is nearly tied on average at q=4 (`−0.002`) and lower at q=6 (`−0.049`).

MerLin therefore does not improve the primary metric on average over the preserved QSVM in this grid. This comparison changes only the kernel path within the preserved local preprocessing and SVM protocol. It does not establish a photonic or quantum advantage, and it cannot be used to validate or refute the paper's qubit-BSP claim.

## 5. Conclusion

On the N=500 PneumoniaMNIST pixel surrogate, the preserved QSVM has the highest mean minority-class F1 at q=4 and q=6. However, paired outcomes are mixed, the q=6 gain over tuned RBF is only `+0.003`, and the linear model never collapses. The QSVM training collapse and the historical train/cross-kernel scaling mismatch further prevent a clean interpretation.

These observations do **not** reproduce the paper's systematic quantum-kernel-advantage claim. They characterize how the preserved upstream pipeline behaves after the minimum CPU and open-data adaptation.

In the matched fourth-model comparison, MerLin has lower mean minority F1 than QSVM at both q values. It is close to the classical baselines at q=4 and lower than them at q=6. The matched MerLin comparison is a photonic adaptation, not additional evidence for or against the paper's qubit-BSP claim. A future corrected protocol must be separately named and must not overwrite this preserved diagnostic baseline.